In [1]:
import os
import glob
import random
import numpy as np
import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import silhouette_score, pairwise_distances

# 1. HYPERPARAMETER & TUNING CONFIGURATION

In [2]:
config = {
    "backbone_name": "resnet18",   # Options: "baseline_cnn", "resnet18", "efficientnet_b0"
    "embedding_dim": 128,          # Target dimensions for Isar DB
    "sample_rate": 16000,          # Standardized downsample rate
    "duration_seconds": 30,        # Analysis window per track
    "learning_rate": 1e-4,         # Tuning range: [1e-5, 1e-3]
    "weight_decay": 1e-4,
    "batch_size": 64,              # Kept modest for triplet grouping stability
    "triplet_margin": 0.3,         # Tuning range: [0.1, 0.5]
    "epochs": 15,
    "device": "cuda" if torch.cuda.is_available() else "cpu"
}

print(f"Execution Device: {config['device'].upper()}")

Execution Device: CUDA


# 2. DIGITAL SIGNAL PROCESSING (DSP) DATA LAYER

In [3]:
class MusicSpectrogramDataset(Dataset):
    def __init__(self, file_paths, labels, target_sample_rate=16000, duration_seconds=10):
        self.file_paths = file_paths
        self.labels = labels
        self.target_sample_rate = target_sample_rate
        self.target_samples = target_sample_rate * duration_seconds 
        
        # Log-Mel Spectrogram Transformation Configuration
        self.mel_spectrogram = T.MelSpectrogram(
            sample_rate=target_sample_rate,
            n_fft=1024,
            hop_length=512,
            n_mels=64
        )
        self.amplitude_to_db = T.AmplitudeToDB()

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        audio_path = self.file_paths[idx]
        label = self.labels[idx]
        
        try:
            waveform, sample_rate = torchaudio.load(audio_path)
            
            # Convert Stereo to Mono
            if waveform.shape[0] > 1:
                waveform = torch.mean(waveform, dim=0, keepdim=True)
                
            # Downsample to target sample rate
            if sample_rate != self.target_sample_rate:
                resampler = T.Resample(orig_freq=sample_rate, new_freq=self.target_sample_rate)
                waveform = resampler(waveform)
                
            # Pad or crop track to match explicit window length precisely
            if waveform.shape[1] > self.target_samples:
                waveform = waveform[:, :self.target_samples]
            elif waveform.shape[1] < self.target_samples:
                padding = self.target_samples - waveform.shape[1]
                waveform = torch.nn.functional.pad(waveform, (0, padding))
                
            # Compute Log-Mel Spectrogram
            mel_spec = self.mel_spectrogram(waveform)
            mel_spec_db = self.amplitude_to_db(mel_spec)
            
            return mel_spec_db, label
            
        except Exception as e:
            # Fallback for occasionally corrupted audio file frames
            return torch.zeros((1, 64, 313)), label

# 3. TRIPLET SAMPLING WRAPPER (METRIC LEARNING PREPARATION)

In [4]:
class TripletMusicDataset(Dataset):
    """
    Wraps standard classification dataset to output triplets on-the-fly:
    (Anchor, Positive, Negative)
    """
    def __init__(self, base_dataset):
        self.base_dataset = base_dataset
        self.labels = np.array(base_dataset.labels)
        self.label_to_indices = {label: np.where(self.labels == label)[0] for label in set(self.labels)}

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, index):
        # Anchor
        anchor_img, anchor_label = self.base_dataset[index]
        
        # Positive (Same label/genre, different file index)
        pos_indices = self.label_to_indices[anchor_label]
        pos_index = index
        while pos_index == index:
            pos_index = random.choice(pos_indices)
        pos_img, _ = self.base_dataset[pos_index]
        
        # Negative (Radically different acoustic genre)
        all_labels = set(self.label_to_indices.keys())
        neg_labels = list(all_labels - {anchor_label})
        neg_label = random.choice(neg_labels)
        neg_index = random.choice(self.label_to_indices[neg_label])
        neg_img, _ = self.base_dataset[neg_index]
        
        return anchor_img, pos_img, neg_img

# 4. AGNOSTIC METRIC ARCHITECTURE FACTORY

In [5]:
class AgnosticMetricNet(nn.Module):
    def __init__(self, backbone_name="resnet18", embedding_dim=128):
        super(AgnosticMetricNet, self).__init__()
        import torchvision.models as torchvision_models
        self.backbone_name = backbone_name
        
        # Standardize 1-channel mono spectrogram matrix to a 3-channel input 
        self.input_adapter = nn.Conv2d(1, 3, kernel_size=1, stride=1, padding=0, bias=False)
        
        if backbone_name == "baseline_cnn":
            self.backbone = nn.Sequential(
                nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
                nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
                nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
                nn.AdaptiveAvgPool2d((1, 1))
            )
            feature_dim = 128
            
        elif backbone_name == "resnet18":
            resnet = torchvision_models.resnet18(pretrained=False)
            feature_dim = resnet.fc.in_features
            self.backbone = nn.Sequential(*list(resnet.children())[:-1])
            
        elif backbone_name == "efficientnet_b0":
            effnet = torchvision_models.efficientnet_b0(pretrained=False)
            feature_dim = effnet.classifier[1].in_features
            self.backbone = nn.Sequential(*list(effnet.features), nn.AdaptiveAvgPool2d(1))
            
        else:
            raise ValueError(f"Unsupported architecture layout configuration: {backbone_name}")
            
        # Projection head to compress latent features into hypersphere embedding limits
        self.projection_head = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.ReLU(),
            nn.Linear(256, embedding_dim)
        )

    def forward(self, x):
        if x.shape[1] == 1:
            x = self.input_adapter(x)
        features = self.backbone(x)
        features = torch.flatten(features, 1)
        embeddings = self.projection_head(features)
        # Apply strict L2 Normalization for predictable Cosine Similarity metrics
        return torch.nn.functional.normalize(embeddings, p=2, dim=1)

# 5. UNIFIED VALIDATION PIPELINE FUNCTION (3-IN-1 COHERENCE CALC)

In [6]:
def evaluate_coherence(model, evaluation_loader, device, k=5):
    """
    Runs validation features to extract a robust 3-axis geometric evaluation matrix.
    Metrics: KNN Purity, Normalized Silhouette, and Mean Average Precision @ K (mAP@K)
    """
    model.eval()
    all_embeddings = []
    all_labels = []
    
    with torch.no_grad():
        for imgs, labels in evaluation_loader:
            imgs = imgs.to(device)
            embeddings = model(imgs)
            all_embeddings.append(embeddings.cpu().numpy())
            all_labels.append(labels.numpy())
            
    all_embeddings = np.concatenate(all_embeddings, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    
    num_samples = all_embeddings.shape[0]
    
    # ---------------------------------------------------------
    # 1. Compute KNN Neighborhood Purity Accuracy
    # ---------------------------------------------------------
    knn = KNeighborsClassifier(n_neighbors=k, metric="cosine")
    knn.fit(all_embeddings, all_labels)
    knn_purity = knn.score(all_embeddings, all_labels)
    
    # ---------------------------------------------------------
    # 2. Compute Global Silhouette Cluster Separation Metric
    # ---------------------------------------------------------
    try:
        sil_score = silhouette_score(all_embeddings, all_labels, metric="cosine")
        normalized_silhouette = (sil_score + 1.0) / 2.0
    except:
        normalized_silhouette = 0.0
        
    # ---------------------------------------------------------
    # 3. Compute Mean Average Precision @ K (mAP@K)
    # ---------------------------------------------------------
    # Calculate a full pairwise distance matrix (Cosine Distance)
    dist_matrix = pairwise_distances(all_embeddings, metric="cosine")
    
    # Sort indices of neighbors for every single track (row) from closest to furthest
    # We exclude the first column [:, 1:k+1] because index 0 is always the track matching itself
    sorted_neighbors = np.argsort(dist_matrix, axis=1)[:, 1:k+1]
    
    map_at_k_sum = 0.0
    
    for i in range(num_samples):
        target_label = all_labels[i]
        neighbor_labels = all_labels[sorted_neighbors[i]]
        
        # Binary array: 1 if neighbor matches target genre, 0 otherwise
        correct_matches = (neighbor_labels == target_label).astype(float)
        
        # Calculate running precision values at each step of K
        precisions = []
        num_correct = 0.0
        for rank in range(k):
            if correct_matches[rank] == 1.0:
                num_correct += 1.0
                precisions.append(num_correct / (rank + 1.0))
        
        # Average Precision for this query track
        if precisions:
            map_at_k_sum += np.mean(precisions)
            
    map_at_k = map_at_k_sum / num_samples

    # ---------------------------------------------------------
    # 4. Harmonized F-Score Variant / Balanced Index
    # ---------------------------------------------------------
    # We use an equal balance or a harmonic mean across all 3 vectors 
    # to penalize the system heavily if even one metric underperforms.
    epsilon = 1e-10
    composite_coherence = 3 / ((1 / (knn_purity + epsilon)) + 
                               (1 / (normalized_silhouette + epsilon)) + 
                               (1 / (map_at_k + epsilon)))
    
    return knn_purity, normalized_silhouette, map_at_k, composite_coherence

# 6. PIPELINE ORCHESTRATION & TRAINING LOOP

In [7]:
if __name__ == "__main__":
    # Mount and map standard GTZAN folder array layout cleanly inside Kaggle
    gtzan_dataset_path = "/kaggle/input/datasets/andradaolteanu/gtzan-dataset-music-genre-classification/Data/genres_original"
    
    if not os.path.exists(gtzan_dataset_path):
        print("⚠️ Execution stopped: Add the GTZAN Dataset asset to your Kaggle Notebook sidebar workspace environment.")
    else:
        file_paths = []
        labels = []
        
        # Enumerate directories to dynamically structure textual classes to integer hashes
        genres = sorted(os.listdir(gtzan_dataset_path))
        genre_to_idx = {genre: idx for idx, genre in enumerate(genres)}
        
        for genre in genres:
            genre_folder = os.path.join(gtzan_dataset_path, genre)
            if os.path.isdir(genre_folder):
                for file in os.listdir(genre_folder):
                    if file.endswith(".wav"):
                        file_paths.append(os.path.join(genre_folder, file))
                        labels.append(genre_to_idx[genre])
                        
        # Train / Validation Split 
        dataset_size = len(file_paths)
        indices = list(range(dataset_size))
        random.seed(42)
        random.shuffle(indices)
        
        split = int(np.floor(0.2 * dataset_size)) # 80-20 partition rule
        train_idx, val_idx = indices[split:], indices[:split]
        
        # Instantiate Dataset layers
        base_train_set = MusicSpectrogramDataset([file_paths[i] for i in train_idx], [labels[i] for i in train_idx])
        base_val_set = MusicSpectrogramDataset([file_paths[i] for i in val_idx], [labels[i] for i in val_idx])
        
        triplet_train_dataset = TripletMusicDataset(base_train_set)
        
        # Loaders configuration
        train_loader = DataLoader(triplet_train_dataset, batch_size=config["batch_size"], shuffle=True, drop_last=True)
        val_eval_loader = DataLoader(base_val_set, batch_size=config["batch_size"], shuffle=False)
        
        # Build Model Factory
        model = AgnosticMetricNet(backbone_name=config["backbone_name"], embedding_dim=config["embedding_dim"])
        model = model.to(config["device"])
        
        criterion = nn.TripletMarginLoss(margin=config["triplet_margin"], p=2)
        optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, 
            T_max=config["epochs"], 
            eta_min=1e-6
        )
        
        print(f"\nStarting Ablation Run Matrix Profile Strategy: {config['backbone_name'].upper()}")
        print("------------------------------------------------------------------------")
        
        for epoch in range(1, config["epochs"] + 1):
            model.train()
            running_loss = 0.0
            current_lr = optimizer.param_groups[0]['lr']
            
            for batch_idx, (anc, pos, neg) in enumerate(train_loader):
                anc, pos, neg = anc.to(config["device"]), pos.to(config["device"]), neg.to(config["device"])
                
                optimizer.zero_grad()
                
                anc_embed = model(anc)
                pos_embed = model(pos)
                neg_embed = model(neg)
                
                loss = criterion(anc_embed, pos_embed, neg_embed)
                loss.backward()
                optimizer.step()
                
                running_loss += loss.item()

            epoch_loss = running_loss / len(train_loader)
            # Perform continuous multi-metric geometric evaluation pass
            knn_acc, sil_score, map_k, composite_idx = evaluate_coherence(model, val_eval_loader, config["device"])

            scheduler.step()
            
            print(f"Epoch [{epoch:02d}/{config['epochs']}] -> Loss: {epoch_loss:.4f} | LR: {current_lr:.6f} | KNN: {knn_acc*100:.1f}% | Sil: {sil_score:.3f} | mAP@{config['batch_size'] if 'batch_size' in config else 5}: {map_k:.3f} | Index: {composite_idx:.4f}")
                
        # Save structural checkpoint ready for script compilation down-conversions
        torch.save(model.state_dict(), f"{config['backbone_name']}_metric_model.pt")
        print("\nExecution Complete. Weights artifact successfully generated and cached.")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)



Starting Ablation Run Matrix Profile Strategy: RESNET18
------------------------------------------------------------------------
Epoch [01/15] -> Loss: 0.2152 | LR: 0.000100 | KNN: 59.0% | Sil: 0.452 | mAP@64: 0.486 | Index: 0.5029
Epoch [02/15] -> Loss: 0.1766 | LR: 0.000099 | KNN: 63.0% | Sil: 0.476 | mAP@64: 0.551 | Index: 0.5453
Epoch [03/15] -> Loss: 0.1398 | LR: 0.000096 | KNN: 64.5% | Sil: 0.493 | mAP@64: 0.587 | Index: 0.5678
Epoch [04/15] -> Loss: 0.1110 | LR: 0.000091 | KNN: 65.0% | Sil: 0.495 | mAP@64: 0.585 | Index: 0.5694
Epoch [05/15] -> Loss: 0.1003 | LR: 0.000084 | KNN: 66.5% | Sil: 0.519 | mAP@64: 0.583 | Index: 0.5830
Epoch [06/15] -> Loss: 0.0823 | LR: 0.000075 | KNN: 67.5% | Sil: 0.526 | mAP@64: 0.584 | Index: 0.5888
Epoch [07/15] -> Loss: 0.0666 | LR: 0.000066 | KNN: 68.5% | Sil: 0.529 | mAP@64: 0.604 | Index: 0.5994
Epoch [08/15] -> Loss: 0.0502 | LR: 0.000056 | KNN: 71.0% | Sil: 0.538 | mAP@64: 0.603 | Index: 0.6092
Epoch [09/15] -> Loss: 0.0465 | LR: 0.000045 |